In [ ]:
%run "./helpers/Helper Functions"

In [ ]:
import os
# test_files dir created in params cell with volume path

In [ ]:
# --- Create Widgets ---

dbutils.widgets.text("dbx_catalog", "", "Databricks Catalog")
dbutils.widgets.text("dbx_schema", "", "Databricks Schema")
dbutils.widgets.text("dbx_volume", "", "Volume name")
dbutils.widgets.text("scale", "100", "Scale Factor (GB)")
dbutils.widgets.dropdown("dbx_optimize_strategy", "liquid_clustering", ["zorder","liquid_clustering","iceberg","iceberg_liquid_clustering","iceberg_zordered"], "Volume name")
dbutils.widgets.text("table_prefix", "", "Table Prefix (e.g. M_100_)")

In [ ]:
import os
# --- Retrieve Values, Set Env Vars, and Build Dictionary ---

params = dbutils.widgets.getAll()

for k,v in params.items():
    os.environ[f"W_{k.upper()}"] = v

# --- Validation Check (all required) ---

if params["dbx_catalog"] == "":
    raise RuntimeError("Validation Error: Parameter 'dbx_catalog' is required but is empty.")

if params["dbx_schema"] == "":
    raise RuntimeError("Validation Error: Parameter 'dbx_schema' is required but is empty.")

if params["dbx_volume"] == "":
    raise RuntimeError("Validation Error: Parameter 'dbx_volume' is required but is empty.")

if params["dbx_optimize_strategy"] == "":
    raise RuntimeError("Validation Error: Parameter 'dbx_optimize_strategy' is required but is empty.")

import os
table_prefix = params.get("table_prefix", "").rstrip("_")
volume_base = f"/Volumes/{params['dbx_catalog']}/{params['dbx_schema']}/{params['dbx_volume']}"
conf_dir = f"{volume_base}/test_files/{table_prefix}" if table_prefix else f"{volume_base}/test_files"
os.makedirs(conf_dir, exist_ok=True)
# Clean old files in this prefix dir
for f in os.listdir(conf_dir):
    if f.endswith("-databricks.tsv"):
        os.remove(os.path.join(conf_dir, f))
print(f"Config dir: {conf_dir}")

In [ ]:
# --- Validate Predictive Optimization is enabled for this catalog + schema ---
# PO is not set per-table here (it is not a table property); we rely on it being
# ENABLEd at the metastore / catalog / schema level and inherited by every managed
# table we create. This cell surfaces a clear WARNING in the notebook output if the
# schema in use does NOT resolve to an enabled state.

def _po_state(fqn: str):
    """Return the 'Predictive Optimization' value from DESCRIBE ... EXTENDED, or None."""
    try:
        rows = spark.sql(f"DESCRIBE SCHEMA EXTENDED {fqn}").collect()
        for r in rows:
            if (r["database_description_item"] or "").strip().lower() == "predictive optimization":
                return (r["database_description_value"] or "").strip()
    except Exception as e:
        print(f"  (could not read PO state for {fqn}: {e})")
    return None

_catalog = params["dbx_catalog"]
_schema = params["dbx_schema"]
_schema_fqn = f"{_catalog}.{_schema}"

_po = _po_state(_schema_fqn)
# Effectively enabled when the resolved state begins with "ENABLE" (e.g.
# "ENABLE (inherited from METASTORE ...)"). "DISABLE" or an unresolved "INHERIT"
# with no enabled ancestor is treated as NOT enabled.
_po_enabled = bool(_po) and _po.upper().startswith("ENABLE")

if _po_enabled:
    print(f"✓ Predictive Optimization is enabled for {_schema_fqn}: {_po}")
else:
    print("=" * 78)
    print("⚠️  WARNING: PREDICTIVE OPTIMIZATION IS NOT ENABLED")
    print("=" * 78)
    print(f"  Schema : {_schema_fqn}")
    print(f"  State  : {_po if _po else 'unknown / not reported'}")
    print()
    print("  Column statistics are collected explicitly via manual ANALYZE (all strategies),")
    print("  so stats do NOT depend on PO. However, PO still handles ongoing compaction and")
    print("  (for the liquid_clustering strategies) background clustering maintenance. With PO")
    print("  off, tables may be less compacted/clustered over time and query-phase results")
    print("  could drift. Enabling PO keeps the layout comparable to Snowflake's auto-maintenance.")
    print()
    print("  Fix (enable + inherit down):")
    print(f"    ALTER SCHEMA {_schema_fqn} ENABLE PREDICTIVE OPTIMIZATION;")
    print(f"    -- or at the catalog: ALTER CATALOG {_catalog} ENABLE PREDICTIVE OPTIMIZATION;")
    print("=" * 78)


In [ ]:
# Create source tables over the raw data
sql_ingestion_queries = {
    "create_call_center": "CREATE TABLE IF NOT EXISTS call_center USING DELTA AS SELECT CAST(_c0 AS INT) AS cc_call_center_sk, CAST(_c1 AS CHAR(16)) AS cc_call_center_id, CAST(_c2 AS DATE) AS cc_rec_start_date, CAST(_c3 AS DATE) AS cc_rec_end_date, CAST(_c4 AS INT) AS cc_closed_date_sk, CAST(_c5 AS INT) AS cc_open_date_sk, CAST(_c6 AS VARCHAR(50)) AS cc_name, CAST(_c7 AS VARCHAR(50)) AS cc_class, CAST(_c8 AS INT) AS cc_employees, CAST(_c9 AS INT) AS cc_sq_ft, CAST(_c10 AS CHAR(20)) AS cc_hours, CAST(_c11 AS VARCHAR(40)) AS cc_manager, CAST(_c12 AS INT) AS cc_mkt_id, CAST(_c13 AS VARCHAR(50)) AS cc_mkt_class, CAST(_c14 AS VARCHAR(100)) AS cc_mkt_desc, CAST(_c15 AS VARCHAR(40)) AS cc_market_manager, CAST(_c16 AS INT) AS cc_division, CAST(_c17 AS VARCHAR(50)) AS cc_division_name, CAST(_c18 AS INT) AS cc_company, CAST(_c19 AS VARCHAR(50)) AS cc_company_name, CAST(_c20 AS CHAR(10)) AS cc_street_number, CAST(_c21 AS VARCHAR(60)) AS cc_street_name, CAST(_c22 AS CHAR(15)) AS cc_street_type, CAST(_c23 AS CHAR(10)) AS cc_suite_number, CAST(_c24 AS VARCHAR(60)) AS cc_city, CAST(_c25 AS VARCHAR(30)) AS cc_county, CAST(_c26 AS CHAR(2)) AS cc_state, CAST(_c27 AS CHAR(10)) AS cc_zip, CAST(_c28 AS VARCHAR(20)) AS cc_country, CAST(_c29 AS DECIMAL(5,2)) AS cc_gmt_offset, CAST(_c30 AS DECIMAL(5,2)) AS cc_tax_percentage, CAST(_c31 AS INT) AS empty FROM read_files('/Volumes/' || ':dbx_catalog' || '/' || ':dbx_schema' || '/' || ':dbx_volume' || '/' || ':dbx_scale' || '/call_center/*.dat', format=>'csv', header=>false, delimiter=>'|')",
    "create_catalog_page": "CREATE TABLE IF NOT EXISTS catalog_page USING DELTA AS SELECT CAST(_c0 AS INT) AS cp_catalog_page_sk, CAST(_c1 AS CHAR(16)) AS cp_catalog_page_id, CAST(_c2 AS INT) AS cp_start_date_sk, CAST(_c3 AS INT) AS cp_end_date_sk, CAST(_c4 AS VARCHAR(50)) AS cp_department, CAST(_c5 AS INT) AS cp_catalog_number, CAST(_c6 AS INT) AS cp_catalog_page_number, CAST(_c7 AS VARCHAR(100)) AS cp_description, CAST(_c8 AS VARCHAR(100)) AS cp_type, CAST(_c9 AS INT) AS empty FROM read_files('/Volumes/' || ':dbx_catalog' || '/' || ':dbx_schema' || '/' || ':dbx_volume' || '/' || ':dbx_scale' || '/catalog_page/*.dat', format=>'csv', header=>false, delimiter=>'|')",
    "create_catalog_returns": "CREATE TABLE IF NOT EXISTS catalog_returns USING DELTA AS SELECT CAST(_c0 AS INT) AS cr_returned_date_sk, CAST(_c1 AS INT) AS cr_returned_time_sk, CAST(_c2 AS INT) AS cr_item_sk, CAST(_c3 AS INT) AS cr_refunded_customer_sk, CAST(_c4 AS INT) AS cr_refunded_cdemo_sk, CAST(_c5 AS INT) AS cr_refunded_hdemo_sk, CAST(_c6 AS INT) AS cr_refunded_addr_sk, CAST(_c7 AS INT) AS cr_returning_customer_sk, CAST(_c8 AS INT) AS cr_returning_cdemo_sk, CAST(_c9 AS INT) AS cr_returning_hdemo_sk, CAST(_c10 AS INT) AS cr_returning_addr_sk, CAST(_c11 AS INT) AS cr_call_center_sk, CAST(_c12 AS INT) AS cr_catalog_page_sk, CAST(_c13 AS INT) AS cr_ship_mode_sk, CAST(_c14 AS INT) AS cr_warehouse_sk, CAST(_c15 AS INT) AS cr_reason_sk, CAST(_c16 AS BIGINT) AS cr_order_number, CAST(_c17 AS INT) AS cr_return_quantity, CAST(_c18 AS DECIMAL(7,2)) AS cr_return_amount, CAST(_c19 AS DECIMAL(7,2)) AS cr_return_tax, CAST(_c20 AS DECIMAL(7,2)) AS cr_return_amt_inc_tax, CAST(_c21 AS DECIMAL(7,2)) AS cr_fee, CAST(_c22 AS DECIMAL(7,2)) AS cr_return_ship_cost, CAST(_c23 AS DECIMAL(7,2)) AS cr_refunded_cash, CAST(_c24 AS DECIMAL(7,2)) AS cr_reversed_charge, CAST(_c25 AS DECIMAL(7,2)) AS cr_store_credit, CAST(_c26 AS DECIMAL(7,2)) AS cr_net_loss, CAST(_c27 AS INT) AS empty FROM read_files('/Volumes/' || ':dbx_catalog' || '/' || ':dbx_schema' || '/' || ':dbx_volume' || '/' || ':dbx_scale' || '/catalog_returns/*.dat', format=>'csv', header=>false, delimiter=>'|')",
    "create_catalog_sales": "CREATE TABLE IF NOT EXISTS catalog_sales USING DELTA AS SELECT CAST(_c0 AS INT) AS cs_sold_date_sk, CAST(_c1 AS INT) AS cs_sold_time_sk, CAST(_c2 AS INT) AS cs_ship_date_sk, CAST(_c3 AS INT) AS cs_bill_customer_sk, CAST(_c4 AS INT) AS cs_bill_cdemo_sk, CAST(_c5 AS INT) AS cs_bill_hdemo_sk, CAST(_c6 AS INT) AS cs_bill_addr_sk, CAST(_c7 AS INT) AS cs_ship_customer_sk, CAST(_c8 AS INT) AS cs_ship_cdemo_sk, CAST(_c9 AS INT) AS cs_ship_hdemo_sk, CAST(_c10 AS INT) AS cs_ship_addr_sk, CAST(_c11 AS INT) AS cs_call_center_sk, CAST(_c12 AS INT) AS cs_catalog_page_sk, CAST(_c13 AS INT) AS cs_ship_mode_sk, CAST(_c14 AS INT) AS cs_warehouse_sk, CAST(_c15 AS INT) AS cs_item_sk, CAST(_c16 AS INT) AS cs_promo_sk, CAST(_c17 AS BIGINT) AS cs_order_number, CAST(_c18 AS INT) AS cs_quantity, CAST(_c19 AS DECIMAL(7,2)) AS cs_wholesale_cost, CAST(_c20 AS DECIMAL(7,2)) AS cs_list_price, CAST(_c21 AS DECIMAL(7,2)) AS cs_sales_price, CAST(_c22 AS DECIMAL(7,2)) AS cs_ext_discount_amt, CAST(_c23 AS DECIMAL(7,2)) AS cs_ext_sales_price, CAST(_c24 AS DECIMAL(7,2)) AS cs_ext_wholesale_cost, CAST(_c25 AS DECIMAL(7,2)) AS cs_ext_list_price, CAST(_c26 AS DECIMAL(7,2)) AS cs_ext_tax, CAST(_c27 AS DECIMAL(7,2)) AS cs_coupon_amt, CAST(_c28 AS DECIMAL(7,2)) AS cs_ext_ship_cost, CAST(_c29 AS DECIMAL(7,2)) AS cs_net_paid, CAST(_c30 AS DECIMAL(7,2)) AS cs_net_paid_inc_tax, CAST(_c31 AS DECIMAL(7,2)) AS cs_net_paid_inc_ship, CAST(_c32 AS DECIMAL(7,2)) AS cs_net_paid_inc_ship_tax, CAST(_c33 AS DECIMAL(7,2)) AS cs_net_profit, CAST(_c34 AS INT) AS empty FROM read_files('/Volumes/' || ':dbx_catalog' || '/' || ':dbx_schema' || '/' || ':dbx_volume' || '/' || ':dbx_scale' || '/catalog_sales/*.dat', format=>'csv', header=>false, delimiter=>'|')",
    "create_customer_address": "CREATE TABLE IF NOT EXISTS customer_address USING DELTA AS SELECT CAST(_c0 AS INT) AS ca_address_sk, CAST(_c1 AS CHAR(16)) AS ca_address_id, CAST(_c2 AS CHAR(10)) AS ca_street_number, CAST(_c3 AS VARCHAR(60)) AS ca_street_name, CAST(_c4 AS CHAR(15)) AS ca_street_type, CAST(_c5 AS CHAR(10)) AS ca_suite_number, CAST(_c6 AS VARCHAR(60)) AS ca_city, CAST(_c7 AS VARCHAR(30)) AS ca_county, CAST(_c8 AS CHAR(2)) AS ca_state, CAST(_c9 AS CHAR(10)) AS ca_zip, CAST(_c10 AS VARCHAR(20)) AS ca_country, CAST(_c11 AS DECIMAL(5,2)) AS ca_gmt_offset, CAST(_c12 AS CHAR(20)) AS ca_location_type, CAST(_c13 AS INT) AS empty FROM read_files('/Volumes/' || ':dbx_catalog' || '/' || ':dbx_schema' || '/' || ':dbx_volume' || '/' || ':dbx_scale' || '/customer_address/*.dat', format=>'csv', header=>false, delimiter=>'|')",
    "create_customer_demographics": "CREATE TABLE IF NOT EXISTS customer_demographics USING DELTA AS SELECT CAST(_c0 AS INT) AS cd_demo_sk, CAST(_c1 AS CHAR(1)) AS cd_gender, CAST(_c2 AS CHAR(1)) AS cd_marital_status, CAST(_c3 AS CHAR(20)) AS cd_education_status, CAST(_c4 AS INT) AS cd_purchase_estimate, CAST(_c5 AS CHAR(10)) AS cd_credit_rating, CAST(_c6 AS INT) AS cd_dep_count, CAST(_c7 AS INT) AS cd_dep_employed_count, CAST(_c8 AS INT) AS cd_dep_college_count, CAST(_c9 AS INT) AS empty FROM read_files('/Volumes/' || ':dbx_catalog' || '/' || ':dbx_schema' || '/' || ':dbx_volume' || '/' || ':dbx_scale' || '/customer_demographics/*.dat', format=>'csv', header=>false, delimiter=>'|')",
    "create_customer": "CREATE TABLE IF NOT EXISTS customer USING DELTA AS SELECT CAST(_c0 AS INT) AS c_customer_sk, CAST(_c1 AS CHAR(16)) AS c_customer_id, CAST(_c2 AS INT) AS c_current_cdemo_sk, CAST(_c3 AS INT) AS c_current_hdemo_sk, CAST(_c4 AS INT) AS c_current_addr_sk, CAST(_c5 AS INT) AS c_first_shipto_date_sk, CAST(_c6 AS INT) AS c_first_sales_date_sk, CAST(_c7 AS CHAR(10)) AS c_salutation, CAST(_c8 AS CHAR(20)) AS c_first_name, CAST(_c9 AS CHAR(30)) AS c_last_name, CAST(_c10 AS CHAR(1)) AS c_preferred_cust_flag, CAST(_c11 AS INT) AS c_birth_day, CAST(_c12 AS INT) AS c_birth_month, CAST(_c13 AS INT) AS c_birth_year, CAST(_c14 AS VARCHAR(20)) AS c_birth_country, CAST(_c15 AS CHAR(13)) AS c_login, CAST(_c16 AS CHAR(50)) AS c_email_address, CAST(_c17 AS INT) AS c_last_review_date_sk, CAST(_c18 AS INT) AS empty FROM read_files('/Volumes/' || ':dbx_catalog' || '/' || ':dbx_schema' || '/' || ':dbx_volume' || '/' || ':dbx_scale' || '/customer/*.dat', format=>'csv', header=>false, delimiter=>'|')",
    "create_date_dim": "CREATE TABLE IF NOT EXISTS date_dim USING DELTA AS SELECT CAST(_c0 AS INT) AS d_date_sk, CAST(_c1 AS CHAR(16)) AS d_date_id, CAST(_c2 AS DATE) AS d_date, CAST(_c3 AS INT) AS d_month_seq, CAST(_c4 AS INT) AS d_week_seq, CAST(_c5 AS INT) AS d_quarter_seq, CAST(_c6 AS INT) AS d_year, CAST(_c7 AS INT) AS d_dow, CAST(_c8 AS INT) AS d_moy, CAST(_c9 AS INT) AS d_dom, CAST(_c10 AS INT) AS d_qoy, CAST(_c11 AS INT) AS d_fy_year, CAST(_c12 AS INT) AS d_fy_quarter_seq, CAST(_c13 AS INT) AS d_fy_week_seq, CAST(_c14 AS CHAR(9)) AS d_day_name, CAST(_c15 AS CHAR(6)) AS d_quarter_name, CAST(_c16 AS CHAR(1)) AS d_holiday, CAST(_c17 AS CHAR(1)) AS d_weekend, CAST(_c18 AS CHAR(1)) AS d_following_holiday, CAST(_c19 AS INT) AS d_first_dom, CAST(_c20 AS INT) AS d_last_dom, CAST(_c21 AS INT) AS d_same_day_ly, CAST(_c22 AS INT) AS d_same_day_lq, CAST(_c23 AS CHAR(1)) AS d_current_day, CAST(_c24 AS CHAR(1)) AS d_current_week, CAST(_c25 AS CHAR(1)) AS d_current_month, CAST(_c26 AS CHAR(1)) AS d_current_quarter, CAST(_c27 AS CHAR(1)) AS d_current_year, CAST(_c28 AS INT) AS empty FROM read_files('/Volumes/' || ':dbx_catalog' || '/' || ':dbx_schema' || '/' || ':dbx_volume' || '/' || ':dbx_scale' || '/date_dim/*.dat', format=>'csv', header=>false, delimiter=>'|')",
    "create_household_demographics": "CREATE TABLE IF NOT EXISTS household_demographics USING DELTA AS SELECT CAST(_c0 AS INT) AS hd_demo_sk, CAST(_c1 AS INT) AS hd_income_band_sk, CAST(_c2 AS CHAR(15)) AS hd_buy_potential, CAST(_c3 AS INT) AS hd_dep_count, CAST(_c4 AS INT) AS hd_vehicle_count, CAST(_c5 AS INT) AS empty FROM read_files('/Volumes/' || ':dbx_catalog' || '/' || ':dbx_schema' || '/' || ':dbx_volume' || '/' || ':dbx_scale' || '/household_demographics/*.dat', format=>'csv', header=>false, delimiter=>'|')",
    "create_income_band": "CREATE TABLE IF NOT EXISTS income_band USING DELTA AS SELECT CAST(_c0 AS INT) AS ib_income_band_sk, CAST(_c1 AS INT) AS ib_lower_bound, CAST(_c2 AS INT) AS ib_upper_bound, CAST(_c3 AS INT) AS empty FROM read_files('/Volumes/' || ':dbx_catalog' || '/' || ':dbx_schema' || '/' || ':dbx_volume' || '/' || ':dbx_scale' || '/income_band/*.dat', format=>'csv', header=>false, delimiter=>'|')",
    "create_inventory": "CREATE TABLE IF NOT EXISTS inventory USING DELTA AS SELECT CAST(_c0 AS INT) AS inv_date_sk, CAST(_c1 AS INT) AS inv_item_sk, CAST(_c2 AS INT) AS inv_warehouse_sk, CAST(_c3 AS INT) AS inv_quantity_on_hand, CAST(_c4 AS INT) AS empty FROM read_files('/Volumes/' || ':dbx_catalog' || '/' || ':dbx_schema' || '/' || ':dbx_volume' || '/' || ':dbx_scale' || '/inventory/*.dat', format=>'csv', header=>false, delimiter=>'|')",
    "create_item": "CREATE TABLE IF NOT EXISTS item USING DELTA AS SELECT CAST(_c0 AS INT) AS i_item_sk, CAST(_c1 AS CHAR(16)) AS i_item_id, CAST(_c2 AS DATE) AS i_rec_start_date, CAST(_c3 AS DATE) AS i_rec_end_date, CAST(_c4 AS VARCHAR(200)) AS i_item_desc, CAST(_c5 AS DECIMAL(7,2)) AS i_current_price, CAST(_c6 AS DECIMAL(7,2)) AS i_wholesale_cost, CAST(_c7 AS INT) AS i_brand_id, CAST(_c8 AS CHAR(50)) AS i_brand, CAST(_c9 AS INT) AS i_class_id, CAST(_c10 AS CHAR(50)) AS i_class, CAST(_c11 AS INT) AS i_category_id, CAST(_c12 AS CHAR(50)) AS i_category, CAST(_c13 AS INT) AS i_manufact_id, CAST(_c14 AS CHAR(50)) AS i_manufact, CAST(_c15 AS CHAR(20)) AS i_size, CAST(_c16 AS CHAR(20)) AS i_formulation, CAST(_c17 AS CHAR(20)) AS i_color, CAST(_c18 AS CHAR(10)) AS i_units, CAST(_c19 AS CHAR(10)) AS i_container, CAST(_c20 AS INT) AS i_manager_id, CAST(_c21 AS CHAR(50)) AS i_product_name, CAST(_c22 AS INT) AS empty FROM read_files('/Volumes/' || ':dbx_catalog' || '/' || ':dbx_schema' || '/' || ':dbx_volume' || '/' || ':dbx_scale' || '/item/*.dat', format=>'csv', header=>false, delimiter=>'|')",
    "create_promotion": "CREATE TABLE IF NOT EXISTS promotion USING DELTA AS SELECT CAST(_c0 AS INT) AS p_promo_sk, CAST(_c1 AS CHAR(16)) AS p_promo_id, CAST(_c2 AS INT) AS p_start_date_sk, CAST(_c3 AS INT) AS p_end_date_sk, CAST(_c4 AS INT) AS p_item_sk, CAST(_c5 AS DECIMAL(15,2)) AS p_cost, CAST(_c6 AS INT) AS p_response_target, CAST(_c7 AS CHAR(50)) AS p_promo_name, CAST(_c8 AS CHAR(1)) AS p_channel_dmail, CAST(_c9 AS CHAR(1)) AS p_channel_email, CAST(_c10 AS CHAR(1)) AS p_channel_catalog, CAST(_c11 AS CHAR(1)) AS p_channel_tv, CAST(_c12 AS CHAR(1)) AS p_channel_radio, CAST(_c13 AS CHAR(1)) AS p_channel_press, CAST(_c14 AS CHAR(1)) AS p_channel_event, CAST(_c15 AS CHAR(1)) AS p_channel_demo, CAST(_c16 AS VARCHAR(100)) AS p_channel_details, CAST(_c17 AS CHAR(15)) AS p_purpose, CAST(_c18 AS CHAR(1)) AS p_discount_active, CAST(_c19 AS INT) AS empty FROM read_files('/Volumes/' || ':dbx_catalog' || '/' || ':dbx_schema' || '/' || ':dbx_volume' || '/' || ':dbx_scale' || '/promotion/*.dat', format=>'csv', header=>false, delimiter=>'|')",
    "create_reason": "CREATE TABLE IF NOT EXISTS reason USING DELTA AS SELECT CAST(_c0 AS INT) AS r_reason_sk, CAST(_c1 AS CHAR(16)) AS r_reason_id, CAST(_c2 AS CHAR(100)) AS r_reason_desc, CAST(_c3 AS INT) AS empty FROM read_files('/Volumes/' || ':dbx_catalog' || '/' || ':dbx_schema' || '/' || ':dbx_volume' || '/' || ':dbx_scale' || '/reason/*.dat', format=>'csv', header=>false, delimiter=>'|')",
    "create_ship_mode": "CREATE TABLE IF NOT EXISTS ship_mode USING DELTA AS SELECT CAST(_c0 AS INT) AS sm_ship_mode_sk, CAST(_c1 AS CHAR(16)) AS sm_ship_mode_id, CAST(_c2 AS CHAR(30)) AS sm_type, CAST(_c3 AS CHAR(10)) AS sm_code, CAST(_c4 AS CHAR(20)) AS sm_carrier, CAST(_c5 AS CHAR(20)) AS sm_contract, CAST(_c6 AS INT) AS empty FROM read_files('/Volumes/' || ':dbx_catalog' || '/' || ':dbx_schema' || '/' || ':dbx_volume' || '/' || ':dbx_scale' || '/ship_mode/*.dat', format=>'csv', header=>false, delimiter=>'|')",
    "create_store_returns": "CREATE TABLE IF NOT EXISTS store_returns USING DELTA AS SELECT CAST(_c0 AS INT) AS sr_returned_date_sk, CAST(_c1 AS INT) AS sr_return_time_sk, CAST(_c2 AS INT) AS sr_item_sk, CAST(_c3 AS INT) AS sr_customer_sk, CAST(_c4 AS INT) AS sr_cdemo_sk, CAST(_c5 AS INT) AS sr_hdemo_sk, CAST(_c6 AS INT) AS sr_addr_sk, CAST(_c7 AS INT) AS sr_store_sk, CAST(_c8 AS INT) AS sr_reason_sk, CAST(_c9 AS BIGINT) AS sr_ticket_number, CAST(_c10 AS INT) AS sr_return_quantity, CAST(_c11 AS DECIMAL(7,2)) AS sr_return_amt, CAST(_c12 AS DECIMAL(7,2)) AS sr_return_tax, CAST(_c13 AS DECIMAL(7,2)) AS sr_return_amt_inc_tax, CAST(_c14 AS DECIMAL(7,2)) AS sr_fee, CAST(_c15 AS DECIMAL(7,2)) AS sr_return_ship_cost, CAST(_c16 AS DECIMAL(7,2)) AS sr_refunded_cash, CAST(_c17 AS DECIMAL(7,2)) AS sr_reversed_charge, CAST(_c18 AS DECIMAL(7,2)) AS sr_store_credit, CAST(_c19 AS DECIMAL(7,2)) AS sr_net_loss, CAST(_c20 AS INT) AS empty FROM read_files('/Volumes/' || ':dbx_catalog' || '/' || ':dbx_schema' || '/' || ':dbx_volume' || '/' || ':dbx_scale' || '/store_returns/*.dat', format=>'csv', header=>false, delimiter=>'|')",
    "create_store_sales": "CREATE TABLE IF NOT EXISTS store_sales USING DELTA AS SELECT CAST(_c0 AS INT) AS ss_sold_date_sk, CAST(_c1 AS INT) AS ss_sold_time_sk, CAST(_c2 AS INT) AS ss_item_sk, CAST(_c3 AS INT) AS ss_customer_sk, CAST(_c4 AS INT) AS ss_cdemo_sk, CAST(_c5 AS INT) AS ss_hdemo_sk, CAST(_c6 AS INT) AS ss_addr_sk, CAST(_c7 AS INT) AS ss_store_sk, CAST(_c8 AS INT) AS ss_promo_sk, CAST(_c9 AS BIGINT) AS ss_ticket_number, CAST(_c10 AS INT) AS ss_quantity, CAST(_c11 AS DECIMAL(7,2)) AS ss_wholesale_cost, CAST(_c12 AS DECIMAL(7,2)) AS ss_list_price, CAST(_c13 AS DECIMAL(7,2)) AS ss_sales_price, CAST(_c14 AS DECIMAL(7,2)) AS ss_ext_discount_amt, CAST(_c15 AS DECIMAL(7,2)) AS ss_ext_sales_price, CAST(_c16 AS DECIMAL(7,2)) AS ss_ext_wholesale_cost, CAST(_c17 AS DECIMAL(7,2)) AS ss_ext_list_price, CAST(_c18 AS DECIMAL(7,2)) AS ss_ext_tax, CAST(_c19 AS DECIMAL(7,2)) AS ss_coupon_amt, CAST(_c20 AS DECIMAL(7,2)) AS ss_net_paid, CAST(_c21 AS DECIMAL(7,2)) AS ss_net_paid_inc_tax, CAST(_c22 AS DECIMAL(7,2)) AS ss_net_profit, CAST(_c23 AS INT) AS empty FROM read_files('/Volumes/' || ':dbx_catalog' || '/' || ':dbx_schema' || '/' || ':dbx_volume' || '/' || ':dbx_scale' || '/store_sales/*.dat', format=>'csv', header=>false, delimiter=>'|')",
    "create_store": "CREATE TABLE IF NOT EXISTS store USING DELTA AS SELECT CAST(_c0 AS INT) AS s_store_sk, CAST(_c1 AS CHAR(16)) AS s_store_id, CAST(_c2 AS DATE) AS s_rec_start_date, CAST(_c3 AS DATE) AS s_rec_end_date, CAST(_c4 AS INT) AS s_closed_date_sk, CAST(_c5 AS VARCHAR(50)) AS s_store_name, CAST(_c6 AS INT) AS s_number_employees, CAST(_c7 AS INT) AS s_floor_space, CAST(_c8 AS CHAR(20)) AS s_hours, CAST(_c9 AS VARCHAR(40)) AS s_manager, CAST(_c10 AS INT) AS s_market_id, CAST(_c11 AS VARCHAR(100)) AS s_geography_class, CAST(_c12 AS VARCHAR(100)) AS s_market_desc, CAST(_c13 AS VARCHAR(40)) AS s_market_manager, CAST(_c14 AS INT) AS s_division_id, CAST(_c15 AS VARCHAR(50)) AS s_division_name, CAST(_c16 AS INT) AS s_company_id, CAST(_c17 AS VARCHAR(50)) AS s_company_name, CAST(_c18 AS VARCHAR(10)) AS s_street_number, CAST(_c19 AS VARCHAR(60)) AS s_street_name, CAST(_c20 AS CHAR(15)) AS s_street_type, CAST(_c21 AS CHAR(10)) AS s_suite_number, CAST(_c22 AS VARCHAR(60)) AS s_city, CAST(_c23 AS VARCHAR(30)) AS s_county, CAST(_c24 AS CHAR(2)) AS s_state, CAST(_c25 AS CHAR(10)) AS s_zip, CAST(_c26 AS VARCHAR(20)) AS s_country, CAST(_c27 AS DECIMAL(5,2)) AS s_gmt_offset, CAST(_c28 AS DECIMAL(5,2)) AS s_tax_precentage, CAST(_c29 AS INT) AS empty FROM read_files('/Volumes/' || ':dbx_catalog' || '/' || ':dbx_schema' || '/' || ':dbx_volume' || '/' || ':dbx_scale' || '/store/*.dat', format=>'csv', header=>false, delimiter=>'|')",
    "create_time_dim": "CREATE TABLE IF NOT EXISTS time_dim USING DELTA AS SELECT CAST(_c0 AS INT) AS t_time_sk, CAST(_c1 AS CHAR(16)) AS t_time_id, CAST(_c2 AS INT) AS t_time, CAST(_c3 AS INT) AS t_hour, CAST(_c4 AS INT) AS t_minute, CAST(_c5 AS INT) AS t_second, CAST(_c6 AS CHAR(2)) AS t_am_pm, CAST(_c7 AS CHAR(20)) AS t_shift, CAST(_c8 AS CHAR(20)) AS t_sub_shift, CAST(_c9 AS CHAR(20)) AS t_meal_time, CAST(_c10 AS INT) AS empty FROM read_files('/Volumes/' || ':dbx_catalog' || '/' || ':dbx_schema' || '/' || ':dbx_volume' || '/' || ':dbx_scale' || '/time_dim/*.dat', format=>'csv', header=>false, delimiter=>'|')",
    "create_warehouse": "CREATE TABLE IF NOT EXISTS warehouse USING DELTA AS SELECT CAST(_c0 AS INT) AS w_warehouse_sk, CAST(_c1 AS CHAR(16)) AS w_warehouse_id, CAST(_c2 AS VARCHAR(20)) AS w_warehouse_name, CAST(_c3 AS INT) AS w_warehouse_sq_ft, CAST(_c4 AS CHAR(10)) AS w_street_number, CAST(_c5 AS VARCHAR(60)) AS w_street_name, CAST(_c6 AS CHAR(15)) AS w_street_type, CAST(_c7 AS CHAR(10)) AS w_suite_number, CAST(_c8 AS VARCHAR(60)) AS w_city, CAST(_c9 AS VARCHAR(30)) AS w_county, CAST(_c10 AS CHAR(2)) AS w_state, CAST(_c11 AS CHAR(10)) AS w_zip, CAST(_c12 AS VARCHAR(20)) AS w_country, CAST(_c13 AS DECIMAL(5,2)) AS w_gmt_offset, CAST(_c14 AS INT) AS empty FROM read_files('/Volumes/' || ':dbx_catalog' || '/' || ':dbx_schema' || '/' || ':dbx_volume' || '/' || ':dbx_scale' || '/warehouse/*.dat', format=>'csv', header=>false, delimiter=>'|')",
    "create_web_page": "CREATE TABLE IF NOT EXISTS web_page USING DELTA AS SELECT CAST(_c0 AS INT) AS wp_web_page_sk, CAST(_c1 AS CHAR(16)) AS wp_web_page_id, CAST(_c2 AS DATE) AS wp_rec_start_date, CAST(_c3 AS DATE) AS wp_rec_end_date, CAST(_c4 AS INT) AS wp_creation_date_sk, CAST(_c5 AS INT) AS wp_access_date_sk, CAST(_c6 AS CHAR(1)) AS wp_autogen_flag, CAST(_c7 AS INT) AS wp_customer_sk, CAST(_c8 AS VARCHAR(100)) AS wp_url, CAST(_c9 AS CHAR(50)) AS wp_type, CAST(_c10 AS INT) AS wp_char_count, CAST(_c11 AS INT) AS wp_link_count, CAST(_c12 AS INT) AS wp_image_count, CAST(_c13 AS INT) AS wp_max_ad_count, CAST(_c14 AS INT) AS empty FROM read_files('/Volumes/' || ':dbx_catalog' || '/' || ':dbx_schema' || '/' || ':dbx_volume' || '/' || ':dbx_scale' || '/web_page/*.dat', format=>'csv', header=>false, delimiter=>'|')",
    "create_web_returns": "CREATE TABLE IF NOT EXISTS web_returns USING DELTA AS SELECT CAST(_c0 AS INT) AS wr_returned_date_sk, CAST(_c1 AS INT) AS wr_returned_time_sk, CAST(_c2 AS INT) AS wr_item_sk, CAST(_c3 AS INT) AS wr_refunded_customer_sk, CAST(_c4 AS INT) AS wr_refunded_cdemo_sk, CAST(_c5 AS INT) AS wr_refunded_hdemo_sk, CAST(_c6 AS INT) AS wr_refunded_addr_sk, CAST(_c7 AS INT) AS wr_returning_customer_sk, CAST(_c8 AS INT) AS wr_returning_cdemo_sk, CAST(_c9 AS INT) AS wr_returning_hdemo_sk, CAST(_c10 AS INT) AS wr_returning_addr_sk, CAST(_c11 AS INT) AS wr_web_page_sk, CAST(_c12 AS INT) AS wr_reason_sk, CAST(_c13 AS BIGINT) AS wr_order_number, CAST(_c14 AS INT) AS wr_return_quantity, CAST(_c15 AS DECIMAL(7,2)) AS wr_return_amt, CAST(_c16 AS DECIMAL(7,2)) AS wr_return_tax, CAST(_c17 AS DECIMAL(7,2)) AS wr_return_amt_inc_tax, CAST(_c18 AS DECIMAL(7,2)) AS wr_fee, CAST(_c19 AS DECIMAL(7,2)) AS wr_return_ship_cost, CAST(_c20 AS DECIMAL(7,2)) AS wr_refunded_cash, CAST(_c21 AS DECIMAL(7,2)) AS wr_reversed_charge, CAST(_c22 AS DECIMAL(7,2)) AS wr_account_credit, CAST(_c23 AS DECIMAL(7,2)) AS wr_net_loss, CAST(_c24 AS INT) AS empty FROM read_files('/Volumes/' || ':dbx_catalog' || '/' || ':dbx_schema' || '/' || ':dbx_volume' || '/' || ':dbx_scale' || '/web_returns/*.dat', format=>'csv', header=>false, delimiter=>'|')",
    "create_web_sales": "CREATE TABLE IF NOT EXISTS web_sales USING DELTA AS SELECT CAST(_c0 AS INT) AS ws_sold_date_sk, CAST(_c1 AS INT) AS ws_sold_time_sk, CAST(_c2 AS INT) AS ws_ship_date_sk, CAST(_c3 AS INT) AS ws_item_sk, CAST(_c4 AS INT) AS ws_bill_customer_sk, CAST(_c5 AS INT) AS ws_bill_cdemo_sk, CAST(_c6 AS INT) AS ws_bill_hdemo_sk, CAST(_c7 AS INT) AS ws_bill_addr_sk, CAST(_c8 AS INT) AS ws_ship_customer_sk, CAST(_c9 AS INT) AS ws_ship_cdemo_sk, CAST(_c10 AS INT) AS ws_ship_hdemo_sk, CAST(_c11 AS INT) AS ws_ship_addr_sk, CAST(_c12 AS INT) AS ws_web_page_sk, CAST(_c13 AS INT) AS ws_web_site_sk, CAST(_c14 AS INT) AS ws_ship_mode_sk, CAST(_c15 AS INT) AS ws_warehouse_sk, CAST(_c16 AS INT) AS ws_promo_sk, CAST(_c17 AS BIGINT) AS ws_order_number, CAST(_c18 AS INT) AS ws_quantity, CAST(_c19 AS DECIMAL(7,2)) AS ws_wholesale_cost, CAST(_c20 AS DECIMAL(7,2)) AS ws_list_price, CAST(_c21 AS DECIMAL(7,2)) AS ws_sales_price, CAST(_c22 AS DECIMAL(7,2)) AS ws_ext_discount_amt, CAST(_c23 AS DECIMAL(7,2)) AS ws_ext_sales_price, CAST(_c24 AS DECIMAL(7,2)) AS ws_ext_wholesale_cost, CAST(_c25 AS DECIMAL(7,2)) AS ws_ext_list_price, CAST(_c26 AS DECIMAL(7,2)) AS ws_ext_tax, CAST(_c27 AS DECIMAL(7,2)) AS ws_coupon_amt, CAST(_c28 AS DECIMAL(7,2)) AS ws_ext_ship_cost, CAST(_c29 AS DECIMAL(7,2)) AS ws_net_paid, CAST(_c30 AS DECIMAL(7,2)) AS ws_net_paid_inc_tax, CAST(_c31 AS DECIMAL(7,2)) AS ws_net_paid_inc_ship, CAST(_c32 AS DECIMAL(7,2)) AS ws_net_paid_inc_ship_tax, CAST(_c33 AS DECIMAL(7,2)) AS ws_net_profit, CAST(_c34 AS INT) AS empty FROM read_files('/Volumes/' || ':dbx_catalog' || '/' || ':dbx_schema' || '/' || ':dbx_volume' || '/' || ':dbx_scale' || '/web_sales/*.dat', format=>'csv', header=>false, delimiter=>'|')",
    "create_web_site": "CREATE TABLE IF NOT EXISTS web_site USING DELTA AS SELECT CAST(_c0 AS INT) AS web_site_sk, CAST(_c1 AS CHAR(16)) AS web_site_id, CAST(_c2 AS DATE) AS web_rec_start_date, CAST(_c3 AS DATE) AS web_rec_end_date, CAST(_c4 AS VARCHAR(50)) AS web_name, CAST(_c5 AS INT) AS web_open_date_sk, CAST(_c6 AS INT) AS web_close_date_sk, CAST(_c7 AS VARCHAR(50)) AS web_class, CAST(_c8 AS VARCHAR(40)) AS web_manager, CAST(_c9 AS INT) AS web_mkt_id, CAST(_c10 AS VARCHAR(50)) AS web_mkt_class, CAST(_c11 AS VARCHAR(100)) AS web_mkt_desc, CAST(_c12 AS VARCHAR(40)) AS web_market_manager, CAST(_c13 AS INT) AS web_company_id, CAST(_c14 AS CHAR(50)) AS web_company_name, CAST(_c15 AS CHAR(10)) AS web_street_number, CAST(_c16 AS VARCHAR(60)) AS web_street_name, CAST(_c17 AS CHAR(15)) AS web_street_type, CAST(_c18 AS CHAR(10)) AS web_suite_number, CAST(_c19 AS VARCHAR(60)) AS web_city, CAST(_c20 AS VARCHAR(30)) AS web_county, CAST(_c21 AS CHAR(2)) AS web_state, CAST(_c22 AS CHAR(10)) AS web_zip, CAST(_c23 AS VARCHAR(20)) AS web_country, CAST(_c24 AS DECIMAL(5,2)) AS web_gmt_offset, CAST(_c25 AS DECIMAL(5,2)) AS web_tax_percentage, CAST(_c26 AS INT) AS empty FROM read_files('/Volumes/' || ':dbx_catalog' || '/' || ':dbx_schema' || '/' || ':dbx_volume' || '/' || ':dbx_scale' || '/web_site/*.dat', format=>'csv', header=>false, delimiter=>'|')",
}

In [ ]:
# Unified clustering/zordering columns per table.
# Used by both zorder and liquid clustering strategies (same columns, same order).
# Based on TPC-DS power test query analysis: join frequency + filter frequency.
# Max 3 columns per table, ordered by usage importance.

clustering_columns = {
    "call_center": ["cc_call_center_sk", "cc_name", "cc_county"],
    "catalog_page": ["cp_catalog_page_sk"],
    "catalog_returns": ["cr_item_sk", "cr_order_number", "cr_returned_date_sk"],
    "catalog_sales": ["cs_sold_date_sk", "cs_item_sk", "cs_bill_customer_sk"],
    "customer": ["c_customer_sk", "c_current_addr_sk", "c_current_cdemo_sk"],
    "customer_address": ["ca_address_sk", "ca_state", "ca_county"],
    "customer_demographics": ["cd_demo_sk", "cd_marital_status", "cd_education_status"],
    "date_dim": ["d_date_sk", "d_year", "d_moy"],
    "household_demographics": ["hd_dep_count", "hd_vehicle_count", "hd_demo_sk"],
    "income_band": ["ib_income_band_sk"],
    "inventory": ["inv_item_sk", "inv_date_sk", "inv_warehouse_sk"],
    "item": ["i_item_sk", "i_category", "i_current_price"],
    "promotion": ["p_promo_sk"],
    "reason": ["r_reason_sk"],
    "ship_mode": ["sm_ship_mode_sk", "sm_carrier"],
    "store": ["s_store_sk", "s_state", "s_zip"],
    "store_returns": ["sr_item_sk", "sr_ticket_number", "sr_customer_sk"],
    "store_sales": ["ss_sold_date_sk", "ss_item_sk", "ss_store_sk"],
    "time_dim": ["t_hour", "t_minute", "t_time_sk"],
    "warehouse": ["w_warehouse_sk"],
    "web_page": ["wp_web_page_sk", "wp_char_count"],
    "web_returns": ["wr_item_sk", "wr_order_number", "wr_returned_date_sk"],
    "web_sales": ["ws_sold_date_sk", "ws_item_sk", "ws_bill_customer_sk"],
    "web_site": ["web_site_sk"],
}

# Build derived dicts from clustering_columns
liquid_clustering_statement_dict = {
    tbl: f"CLUSTER BY ({', '.join(cols)})"
    for tbl, cols in clustering_columns.items()
}

In [ ]:
# OPTIMIZE queries for liquid clustering (no ZORDER needed - clustering handles it)
liquid_clustering_optimize_queries = {
    f"optimize_{tbl}": f"OPTIMIZE {tbl}"
    for tbl in clustering_columns.keys()
}

In [ ]:
# Build OPTIMIZE ZORDER BY queries from clustering_columns
zorder_queries = {
    f"optimize_{tbl}": f"OPTIMIZE {tbl} ZORDER BY ({', '.join(cols)})"
    for tbl, cols in clustering_columns.items()
}

In [ ]:
# ANALYZE TABLE queries
analyze_table_queries = {
    f"analyze_{tbl}": f"ANALYZE TABLE {tbl} COMPUTE STATISTICS FOR ALL COLUMNS"
    for tbl in clustering_columns.keys()
}

In [ ]:
import re


def add_liquid_clustering_clause(create_sql: str, extra: str | None, tbl_props: dict | None = None) -> str:
    """Insert CLUSTER BY and TBLPROPERTIES BEFORE the AS SELECT clause.

    Spark's CTAS grammar requires:
      CREATE TABLE x [USING ...] [CLUSTER BY ...] [TBLPROPERTIES (...)] AS SELECT ...
    Appending these clauses at the end (after FROM read_files(...)) is rejected
    with "PARSE_SYNTAX_ERROR".

    TBLPROPERTIES values are emitted as quoted strings (Spark requires
    \'key\' = \'value\' syntax — not Python-dict {\'key\': value}).
    """
    base = re.sub(r"\s*;\s*$", "", create_sql)
    clauses = []
    if extra:
        clauses.append(extra)
    if tbl_props:
        kv = ", ".join(f"\'{k}\' = \'{str(v).lower()}\'" for k, v in tbl_props.items())
        clauses.append(f"TBLPROPERTIES ({kv})")
    if not clauses:
        return base + ";"
    insertion = " " + " ".join(clauses)
    if " AS SELECT " in base:
        head, _, tail = base.partition(" AS SELECT ")
        return f"{head}{insertion} AS SELECT {tail};"
    # Non-CTAS form (DDL only) — append at end
    return f"{base}{insertion};"


def switch_to_iceberg(create_sql: str) -> str:
    """Change a CTAS from USING DELTA to USING ICEBERG."""
    return create_sql.replace('USING DELTA', 'USING ICEBERG')


strategy = params["dbx_optimize_strategy"].strip().replace(" ", "_")

# Managed Iceberg tables are always created at Iceberg format-version 3 (deletion
# vectors, VARIANT, row lineage). Requires Databricks Runtime 18.0+ to read/write.
ICEBERG_TBL_PROPS = {"format-version": "3"}

if strategy == "liquid_clustering":
    modified = {}
    for name, stmt in sql_ingestion_queries.items():
        tbl_name = name.removeprefix("create_")
        extra = liquid_clustering_statement_dict.get(tbl_name, "")
        # USING DELTA + CLUSTER BY (...). No TBLPROPERTIES: eager clustering is
        # left at its default (enabled), so the CTAS clusters data at write time.
        modified[name] = add_liquid_clustering_clause(stmt, extra, None)
    sql_ingestion_queries = modified

elif strategy == "iceberg":
    modified = {}
    for name, stmt in sql_ingestion_queries.items():
        # USING ICEBERG at format-version 3 (no clustering).
        modified[name] = switch_to_iceberg(add_liquid_clustering_clause(stmt, None, ICEBERG_TBL_PROPS))
    sql_ingestion_queries = modified

elif strategy == "iceberg_liquid_clustering":
    modified = {}
    for name, stmt in sql_ingestion_queries.items():
        tbl_name = name.removeprefix("create_")
        extra = liquid_clustering_statement_dict.get(tbl_name, "")
        # USING ICEBERG + CLUSTER BY (...) at format-version 3. Nothing else to set:
        # v3 already turns row tracking on (iceberg.enableRowTracking=true) and
        # deletion vectors on by default, matching the plain iceberg strategy.
        modified[name] = switch_to_iceberg(add_liquid_clustering_clause(stmt, extra, ICEBERG_TBL_PROPS))
    sql_ingestion_queries = modified
elif strategy == "iceberg_zordered":
    modified = {}
    for name, stmt in sql_ingestion_queries.items():
        # Iceberg CTAS at format-version 3; zorder layout applied post-write via OPTIMIZE ZORDER BY.
        modified[name] = switch_to_iceberg(add_liquid_clustering_clause(stmt, None, ICEBERG_TBL_PROPS))
    sql_ingestion_queries = modified


In [ ]:
sql_ingestion_queries_transformed = {name: q.replace(':dbx_catalog', params["dbx_catalog"]).replace(':dbx_schema', params["dbx_schema"]).replace(':dbx_volume', params["dbx_volume"]).replace(':dbx_scale', params["scale"]) for name, q in sql_ingestion_queries.items()}

strategy = params["dbx_optimize_strategy"].strip().replace(" ", "_")

# --- Manual ANALYZE runs for ALL Databricks strategies ---
# Column statistics are required for good join planning (broadcast vs shuffle).
# Predictive Optimization does run ANALYZE eventually, but ASYNCHRONOUSLY and only
# after some delay, so tests can start against under-analyzed tables. We therefore
# always run ANALYZE ourselves (out-of-band, un-metered) so every strategy is
# measured statistics-ready. ANALYZE is submitted CONCURRENTLY (see 5a runner).
#
# --- Manual OPTIMIZE only for the ZORDER strategies ---
# zorder / iceberg_zordered need an explicit OPTIMIZE ... ZORDER BY to lay out data.
# liquid_clustering / iceberg_liquid_clustering cluster at write (eager) + PO compacts,
# and plain iceberg needs no layout step — so OPTIMIZE is skipped for those.
analyze_sql = dict(analyze_table_queries)   # every strategy

if strategy in ("zorder", "iceberg_zordered"):
    # ZORDER layout applied post-write via OPTIMIZE ... ZORDER BY (sequential).
    optimize_sql = {**zorder_queries}
else:
    # liquid_clustering, iceberg_liquid_clustering, iceberg: no manual OPTIMIZE.
    optimize_sql = {}

# Apply table name prefix if set (e.g. M_100_ -> M_100_store_sales)
table_prefix = params.get("table_prefix", "")
if table_prefix:
    import re as _re
    # For ingestion queries: only prefix the table name in CREATE TABLE, not in read_files paths
    def _prefix_create_table(sql, prefix):
        return _re.sub(
            r'(CREATE TABLE IF NOT EXISTS\s+)(\w+)',
            lambda m: m.group(1) + prefix + m.group(2),
            sql
        )
    sql_ingestion_queries_transformed = {name: _prefix_create_table(stmt, table_prefix) for name, stmt in sql_ingestion_queries_transformed.items()}
    # For optimize/analyze queries: safe to use prefix_table_names (no file paths)
    optimize_sql = {name: prefix_table_names(stmt, table_prefix) for name, stmt in optimize_sql.items()}
    analyze_sql = {name: prefix_table_names(stmt, table_prefix) for name, stmt in analyze_sql.items()}

persist_to_tsv(sql_ingestion_queries_transformed, f'{conf_dir}/tpcds-queries-ingestion-databricks.tsv')
persist_to_tsv(optimize_sql, f'{conf_dir}/tpcds-queries-optimization-databricks.tsv')
persist_to_tsv(analyze_sql, f'{conf_dir}/tpcds-queries-analyze-databricks.tsv')


In [ ]:
import csv as _csv, shutil as _shutil

table_prefix = params.get("table_prefix", "")
if table_prefix:
    rows = []
    with open('./resources/tpcds-queries-databricks.tsv', 'r') as f:
        reader = _csv.reader(f, delimiter='\t', quoting=_csv.QUOTE_NONE)
        for row in reader:
            rows.append(row)
    # Apply prefix to query column (index 1), skip header
    for i in range(1, len(rows)):
        rows[i][1] = prefix_table_names(rows[i][1], table_prefix)
    with open(f'{conf_dir}/tpcds-queries-perf-databricks.tsv', 'w', newline='') as f:
        writer = _csv.writer(f, delimiter='\t', quoting=_csv.QUOTE_NONE, escapechar=None, quotechar=None)
        writer.writerows(rows)
    print(f"Perf queries prefixed with '{table_prefix}'")
else:
    _shutil.copy('./resources/tpcds-queries-databricks.tsv', f'{conf_dir}/tpcds-queries-perf-databricks.tsv')
    print("Perf queries copied (no prefix)")